# HAM10000 EDA（刘继业 2252752）

本 Notebook 用于完成数据工程部分的探索性数据分析，并将图表保存到 `reports/figures/`。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata_clean.csv"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font="Arial")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("未找到 data/processed/metadata_clean.csv，请先运行 python src/data/clean_metadata.py")

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
plt.figure(figsize=(8, 5))
order = df["dx"].value_counts().index
sns.countplot(data=df, x="dx", order=order, color="#4C78A8")
plt.title("HAM10000 Class Distribution")
plt.xlabel("Diagnosis Class")
plt.ylabel("Sample Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["age"], bins=30, kde=True, color="#59A14F")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Sample Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "age_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sex_order = df["sex"].value_counts().index
sns.countplot(data=df, x="sex", order=sex_order, color="#F28E2B")
plt.title("Sex Distribution")
plt.xlabel("Sex")
plt.ylabel("Sample Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sex_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
loc_order = df["localization"].value_counts().index
sns.countplot(data=df, y="localization", order=loc_order, color="#E15759")
plt.title("Localization Distribution")
plt.xlabel("Sample Count")
plt.ylabel("Localization")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "localization_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
dx_type_order = df["dx_type"].value_counts().index
sns.countplot(data=df, x="dx_type", order=dx_type_order, color="#76B7B2")
plt.title("Diagnosis Type Distribution")
plt.xlabel("Diagnosis Type")
plt.ylabel("Sample Count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "dx_type_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
missing_summary = df.isna().sum().sort_values(ascending=False)
sns.barplot(x=missing_summary.index, y=missing_summary.values, color="#B07AA1")
plt.title("Missing Values by Field")
plt.xlabel("Field")
plt.ylabel("Missing Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "missing_values.png", dpi=300)
plt.show()

In [ ]:
classes = sorted(df["dx"].unique())
fig, axes = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3))

for ax, dx in zip(axes, classes):
    row = df[df["dx"] == dx].iloc[0]
    image_path = PROJECT_ROOT / row["image_path"]
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"{dx}\nlabel={row['label']}")
    ax.set_xlabel("Width")
    ax.set_ylabel("Height")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Sample Images by Diagnosis Class", y=1.05)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sample_images_by_class.png", dpi=300, bbox_inches="tight")
plt.show()

## EDA 结论

- HAM10000 存在明显类别不平衡。
- `nv` 类样本最多，是数据集中占比最高的类别。
- 少数类样本较少，训练阶段需要使用类别权重或过采样策略，例如 `WeightedRandomSampler`。
- 年龄、性别、病灶部位可以作为元数据特征加入多模态融合模型。
- 验证集和测试集不能使用随机数据增强，只应使用确定性预处理。